# EEGNet — MLSPred-Bench Results

Loads the JSON written by `train.py` and visualises:
1. Train / val loss curves
2. Accuracy, Sensitivity, Specificity over epochs
3. Test confusion matrix
4. AUROC curve over epochs
5. Final metric summary bar chart
6. Cross-benchmark comparison (if multiple result files exist)

In [ ]:
import json
import glob
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

try:
    import seaborn as sns
    sns.set_theme(style='whitegrid', palette='muted')
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

RESULTS_DIR  = Path('results')
EXPERIMENT   = 'eegnet_bmrk12'   # ← change to match --name used in train.py

with open(RESULTS_DIR / f'{EXPERIMENT}_results.json') as f:
    R = json.load(f)

history  = R['history']
epochs   = list(range(1, len(history['train_loss']) + 1))
test_m   = R['test_metrics']

print(f"Benchmark    : {R['benchmark_id']}  "
      f"(SPH={R['sph_min']} min, SOP={R['sop_min']} min)")
print(f"Best epoch   : {R['best_epoch']}")
print(f"Model params : {R['n_params']:,}")

## 1. Loss Curves

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(epochs, history['train_loss'], label='Train')
ax.plot(epochs, history['val_loss'],   label='Val')
ax.axvline(R['best_epoch'], color='grey', ls='--', alpha=0.6,
           label=f'Best epoch ({R["best_epoch"]})')
ax.set_xlabel('Epoch'); ax.set_ylabel('Cross-entropy loss')
ax.set_title('Training / Validation Loss'); ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / f'{EXPERIMENT}_loss.png', dpi=150)
plt.show()

## 2. Classification Metrics over Epochs

In [ ]:
def get(metric_list, key):
    return [m.get(key, float('nan')) for m in metric_list]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
panels = [
    ('accuracy',    'Accuracy'),
    ('sensitivity', 'Sensitivity (Preictal Recall)'),
    ('specificity', 'Specificity (Interictal Recall)'),
]
for ax, (key, title) in zip(axes, panels):
    ax.plot(epochs, get(history['train_metrics'], key), label='Train')
    ax.plot(epochs, get(history['val_metrics'],   key), label='Val')
    ax.axvline(R['best_epoch'], color='grey', ls='--', alpha=0.6)
    ax.set_ylim(0, 1); ax.set_xlabel('Epoch'); ax.set_title(title)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.legend(fontsize=8)
plt.suptitle('Metrics over Training', y=1.01)
plt.tight_layout()
plt.savefig(RESULTS_DIR / f'{EXPERIMENT}_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Test-Set Confusion Matrix

In [ ]:
cm = np.array([[test_m['tn'], test_m['fp']],
               [test_m['fn'], test_m['tp']]])
fig, ax = plt.subplots(figsize=(5, 4))
if HAS_SEABORN:
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred: Interictal', 'Pred: Preictal'],
                yticklabels=['True: Interictal', 'True: Preictal'])
else:
    im = ax.imshow(cm, cmap='Blues')
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, str(v), ha='center', va='center', fontsize=14)
    ax.set_xticks([0,1]); ax.set_xticklabels(['Pred: Interictal','Pred: Preictal'])
    ax.set_yticks([0,1]); ax.set_yticklabels(['True: Interictal','True: Preictal'])
    plt.colorbar(im, ax=ax)
ax.set_title('Test Set — Confusion Matrix')
plt.tight_layout()
plt.savefig(RESULTS_DIR / f'{EXPERIMENT}_confusion.png', dpi=150)
plt.show()

## 4. AUROC over Epochs

In [ ]:
auroc_vals = get(history['val_metrics'], 'auroc')
if not all(np.isnan(auroc_vals)):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(epochs, auroc_vals, color='steelblue')
    ax.axvline(R['best_epoch'], color='grey', ls='--', alpha=0.6)
    ax.set_ylim(0.5, 1.0); ax.set_xlabel('Epoch'); ax.set_ylabel('AUROC')
    ax.set_title('Validation AUROC over Training')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f'{EXPERIMENT}_auroc.png', dpi=150)
    plt.show()
else:
    print('No AUROC data.')

## 5. Metric Summary Bar Chart

In [ ]:
bar_keys   = ['sensitivity', 'specificity', 'accuracy', 'f1_preictal']
bar_labels = ['Sensitivity', 'Specificity', 'Accuracy',  'F1 Preictal']
bar_vals   = [test_m.get(k, 0.0) for k in bar_keys]

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#2196F3','#4CAF50','#FF9800','#9C27B0']
bars = ax.bar(bar_labels, bar_vals, color=colors, edgecolor='white')
for bar, val in zip(bars, bar_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)
ax.set_ylim(0, 1.12)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_title(f'EEGNet — Benchmark {R["benchmark_id"]} Test Performance')
plt.tight_layout()
plt.savefig(RESULTS_DIR / f'{EXPERIMENT}_summary_bar.png', dpi=150)
plt.show()

print(f"FPR/hour : {test_m.get('fpr_per_hour', 0):.2f}")
if 'auroc' in test_m:
    print(f"AUROC    : {test_m['auroc']:.4f}")

## 6. Cross-Benchmark Comparison
Reads all `eegnet_bmrk*_results.json` files and plots test AUROC vs benchmark ID.

In [ ]:
all_files = sorted(glob.glob(str(RESULTS_DIR / 'eegnet_bmrk*_results.json')))

if len(all_files) < 2:
    print('Run train.py on multiple benchmarks to see a comparison.')
else:
    bmrk_ids, aucs, sens, specs = [], [], [], []
    for fp in all_files:
        with open(fp) as f:
            r = json.load(f)
        bmrk_ids.append(r['benchmark_id'])
        aucs.append(r['test_metrics'].get('auroc', float('nan')))
        sens.append(r['test_metrics'].get('sensitivity', float('nan')))
        specs.append(r['test_metrics'].get('specificity', float('nan')))

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, vals, title in zip(
        axes,
        [aucs, sens, specs],
        ['Test AUROC', 'Test Sensitivity', 'Test Specificity']
    ):
        ax.bar([str(b) for b in bmrk_ids], vals, color='steelblue', alpha=0.85)
        ax.set_xlabel('Benchmark ID'); ax.set_ylabel(title)
        ax.set_title(title); ax.set_ylim(0, 1)
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    plt.suptitle('EEGNet — Cross-Benchmark Comparison', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'eegnet_cross_benchmark.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7. Printed Summary Table

In [ ]:
rows = [
    ('accuracy',        'Accuracy'),
    ('sensitivity',     'Sensitivity (Preictal Recall)'),
    ('specificity',     'Specificity (Interictal Recall)'),
    ('f1_weighted',     'F1 (Weighted)'),
    ('f1_preictal',     'F1 (Preictal)'),
    ('auroc',           'AUROC'),
    ('fpr_per_hour',    'False Positive Rate (/hour)'),
    ('time_in_warning', 'Time in Warning'),
]
print(f"Benchmark {R['benchmark_id']}  "
      f"(SPH={R['sph_min']} min, SOP={R['sop_min']} min)")
print(f"{'Metric':<35} {'Value':>10}")
print('-' * 47)
for key, label in rows:
    if key in test_m:
        print(f'{label:<35} {test_m[key]:>10.4f}')
print(f"{'Best epoch':<35} {R['best_epoch']:>10}")
print(f"{'Model parameters':<35} {R['n_params']:>10,}")